In [1]:
from openai import AsyncOpenAI
#from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import os
from pydantic import BaseModel
from pypdf import PdfReader
import requests # needed to do the push notification
import asyncio
import certifi
from dotenv import load_dotenv
from agents import Agent
load_dotenv(override=True)


True

In [7]:
from agents import Agent
from openai import OpenAI
from agents import trace, Runner, OpenAIChatCompletionsModel


In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-


In [4]:
reader = PdfReader("me/SRP_CV2025v1.pdf")
cv = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        cv += text

reader = PdfReader("me/SRP_resume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [5]:
name = "Sebastian Rueda Parra"
instruction = f"You are acting as a recruitment advisor for people that are looking for jobs. {name} is looking for jobs as data-scientist and AI-engineer\
You are advising a client: {name}, on how to improve their resume, \
he wants to improve the writing in his resume to capture attention of a potential employer and pass AI filters after application. \
As a very important point, {name} desires to include his new experience AI engineering, he has been working with AI workflows and AI agents (including openAI SDK, and Crew AI) \
Your responsibility is to advice {name} for changes to his resume, including the structure, the content, the formatting. \
Use of buzzwords that will improve engagement from possible employers. \
You are given {name}'s (your client) curriculum vitae (CV) and his current resume. \
Use the information in the CV and current resume to suggest for improvements that could be used in a new version of the resume. \
Produce output text that reflexts all of your suggestions in great deatail.\
Provide all the necessary feedback and avoid the use of the word 'leverage', No use of that word would be tolerated."

instruction += f"\n\n## {name}'s CV:\n{cv}\n\n## {name}'s Current Resume:\n{resume}\n\n"



In [9]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)

In [10]:
adv_agent1 = Agent(name="DS_adv_Agent", instructions=instruction, model=deepseek_model)
adv_agent2 = Agent(name="Ge_adv_Agent", instructions=instruction, model=gemini_model)

adv_agent3 = Agent(name="OAP_adv_Agent",instructions=instruction,model="gpt-4o-mini")

In [11]:
message = f"provide suggestions for improving {name}'s resume"
with trace("advisor_agent2"):
    result = await Runner.run(adv_agent2, message)
    print(result.final_output)

Okay, Sebastian, let's overhaul your resume to make it irresistible to potential employers and optimized for both human and AI screening! We'll focus on content, structure, formatting, and the strategic inclusion of relevant keywords.

**Overall Strategy:**

*   **Targeted Resume:** Tailor the resume to each specific job you apply for. While this master resume will be comprehensive, always tweak it to match the job description's required skills and experience.
*   **Quantifiable Achievements:** Whenever possible, use numbers and metrics to demonstrate the impact of your work.
*   **Action Verbs:** Start each bullet point with a strong action verb (e.g., "Developed," "Implemented," "Optimized," "Automated," "Designed").
*   **AI-Friendly Formatting:** Use a clean, ATS-compatible layout. Avoid excessive use of tables, images, or unusual fonts, as these can confuse applicant tracking systems (ATS).
*   **Showcase AI Engineering Experience:** Highlight your recent work with AI Agents and w

In [12]:
message = f"provide suggestions for improving {name}'s resume"
with trace("Parallel_Resume_Advisors"):
    results = await asyncio.gather(
        Runner.run(adv_agent1,message),
        Runner.run(adv_agent2,message),
        Runner.run(adv_agent3,message),
    )

In [13]:
message = f"provide suggestions for improving {name}'s resume"

adv_tool1 = adv_agent1.as_tool(tool_name="D_adv_Agent", tool_description=message)
adv_tool2 = adv_agent2.as_tool(tool_name="G_adv_Agent", tool_description=message)
adv_tool3 = adv_agent3.as_tool(tool_name="O_adv_Agent", tool_description=message)

In [14]:
resume_writer_instructions = f"""
You are 'resume_writer', an AI agent responsible for creating a new, polished resume for {name}.  
You will receive detailed improvement suggestions from another agent named 'adv_manager'.  
Your sole task is to generate a new resume strictly following those suggestions, using information extracted from {name}'s curriculum vitae (CV) and current resume.  

### Instructions
1. Use only the suggestions provided by adv_manager, along with the content from the CV and current resume.  
   - Do not add, modify, or infer information beyond what is explicitly given.  
   - Do not generate your own suggestions or creative content.  

2. Create a **new version** of the resume that:  
   - Fully implements adv_manager’s suggestions.  
   - Maintains professional tone and clarity.  
   - Is well-formatted, concise, and ready for PDF conversion.  
   - Does not exceed one page in length.  

3. Output only the final resume text (no explanations, metadata, or formatting notes).  
   Ensure spacing and layout are clean and appropriate for a one-page PDF.  

---

## {name}'s Curriculum Vitae (CV):
{cv}

---

## {name}'s Current Resume:
{resume}
"""
print(resume_writer_instructions)


You are 'resume_writer', an AI agent responsible for creating a new, polished resume for Sebastian Rueda Parra.  
You will receive detailed improvement suggestions from another agent named 'adv_manager'.  
Your sole task is to generate a new resume strictly following those suggestions, using information extracted from Sebastian Rueda Parra's curriculum vitae (CV) and current resume.  

### Instructions
1. Use only the suggestions provided by adv_manager, along with the content from the CV and current resume.  
   - Do not add, modify, or infer information beyond what is explicitly given.  
   - Do not generate your own suggestions or creative content.  

2. Create a **new version** of the resume that:  
   - Fully implements adv_manager’s suggestions.  
   - Maintains professional tone and clarity.  
   - Is well-formatted, concise, and ready for PDF conversion.  
   - Does not exceed one page in length.  

3. Output only the final resume text (no explanations, metadata, or formatting

In [15]:
writer_agent = Agent(
    name="writer_agent",
    instructions=resume_writer_instructions,
    model="gpt-4o-mini", handoff_description=f"Produce a new version of {name}'s resume")

In [16]:
advtools = [adv_tool1, adv_tool2, adv_tool3]
handoffs = [writer_agent]
print(advtools)
print(handoffs)

[FunctionTool(name='D_adv_Agent', description="provide suggestions for improving Sebastian Rueda Parra's resume", params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7c5b40123770>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='G_adv_Agent', description="provide suggestions for improving Sebastian Rueda Parra's resume", params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'typ

In [17]:
advisor_manager_instructions = f"""
You are an avisor manager for people looking for jobs. Your goal is to combine the best suggestions, using the adv_tools (D_adv_Agent, G_adv_Agent. O_adv_agent) to improve {name}'s Resume. \

Follow these steps carefully:\
1. Generate Suggestions: Use all three the adv_tools (D_adv_Agent, G_adv_Agent. O_adv_agent) to generate three different sets of suggestions to improve {name}'s resume. Do not proceed until all three suggestions are ready.\
 
2. Evaluate and combine: Review the suggestions and combine these suggestions using your judgment of which ones are most effective to help {name} find a job as a data-scientist, or AI engineer.\
You can use the tools multiple times if you're not satisfied with the results from the first try. Do not exceed 2 calls per adv_agent.\
 
3. write a (one) detailed compilation of suggestions for resume improvement from combining the 3 suggestions from the adv_agents. \
 
Crucial Rules:\
- You must use the adv_tools (D_adv_Agent, G_adv_Agent. O_adv_agent) to generate the suggestion drafts — do not write them yourself. No more than 2 calls per agent\
- NEVER use the word 'leverage'. NEVER \
- You must hand off the combined suggestions to the writer_agent  to generate the final resume.
"""

print(advisor_manager_instructions)


You are an avisor manager for people looking for jobs. Your goal is to combine the best suggestions, using the adv_tools (D_adv_Agent, G_adv_Agent. O_adv_agent) to improve Sebastian Rueda Parra's Resume. 
Follow these steps carefully:1. Generate Suggestions: Use all three the adv_tools (D_adv_Agent, G_adv_Agent. O_adv_agent) to generate three different sets of suggestions to improve Sebastian Rueda Parra's resume. Do not proceed until all three suggestions are ready.
2. Evaluate and combine: Review the suggestions and combine these suggestions using your judgment of which ones are most effective to help Sebastian Rueda Parra find a job as a data-scientist, or AI engineer.You can use the tools multiple times if you're not satisfied with the results from the first try. Do not exceed 2 calls per adv_agent.
3. write a (one) detailed compilation of suggestions for resume improvement from combining the 3 suggestions from the adv_agents. 
Crucial Rules:- You must use the adv_tools (D_adv_Age

In [18]:
message = f"produce a consolidated and detailed set of suggestions to improve {name}'s resume"

adv_manager = Agent(
    name="advisor_manager",
    instructions=advisor_manager_instructions,
    tools=advtools,
    handoffs=handoffs,
    model="gpt-4o-mini")

with trace("Automated Resume writer"):
    result = await Runner.run(adv_manager, message)
    print(result.final_output)

```plaintext
Sebastián Rueda Parra  
518-246-0936 | sebastianruedaparra13@gmail.com | linkedin.com/in/srpns | github.com/zevaz13  
Albany, NY, US  

---

**Objective**  
Dynamic Data Scientist and AI Engineer with a strong foundation in neural engineering, robotics, and computer vision. Eager to apply expertise in developing innovative AI workflows and data-driven solutions that enhance healthcare outcomes and automate complex processes. 

---

**Experience**

**Postdoctoral Researcher**  
*National Center for Adaptive Neurotechnologies (NCAN), Albany, NY*  
*Jan 2023 – Present*  
- Engineered an end-to-end Brain-Computer Interface (BCI) system, optimizing color perception assessment with a 99% accuracy in just one minute of data.  
- Designed and implemented machine learning models for biomarker validation in stroke research, enhancing predictive capabilities.  
- Developed comprehensive data pipelines using Arduino, C#, Python, and SQL to streamline experimental control.  
- Automate

## Check out the trace:

https://platform.openai.com/traces